In [1]:
!pip install langchain langchain-groq python-dotenv

In [2]:
import os

os.environ["GROQ_API_KEY"] = "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"

In [3]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=1.0
)

In [4]:
def safe_invoke(prompt):

    try:
        response = llm.invoke(prompt)
        return response.content

    except Exception as e:
        print("Error:", e)
        return ""

In [5]:
def outline_agent(topic):

    prompt = f"""
    Create a detailed article outline
    for the topic:

    {topic}

    Requirements:
    - Around 2500 words total
    - 8 to 10 sections
    - Logical flow
    - Include introduction and conclusion
    - Return only headings
    """

    return safe_invoke(prompt)

In [6]:
def research_agent(topic, heading):

    prompt = f"""
    Topic:
    {topic}

    Section:
    {heading}

    Generate concise research notes
    only for this section.

    Include:
    - important facts
    - examples
    - statistics if relevant
    - latest trends if relevant
    - key concepts

    Keep response concise.
    """

    return safe_invoke(prompt)

In [7]:
def writer_agent(
    topic,
    heading,
    research_notes,
    previous_content=""
):

    prompt = f"""
    You are an expert content writer.

    Topic:
    {topic}

    Current Section:
    {heading}

    Previous Content:
    {previous_content[-1000:]}  #Problem if we send full article.Imagine article already:2000 words.Every new section sends:Full article again then Prompt becomes huge.
                                #Problems:slower, more token cost, API expensive
    {research_notes}

    Write a detailed section.

    Requirements:
    - Write between 320–350 words
    - Never write below 300 words
    - Human-like writing
    - Professional tone
    - Natural flow
    - Original content
    - Avoid repetition
    - Include examples where relevant
    - Do not repeat previous sections
    - Avoid robotic wording
    """

    return safe_invoke(prompt)

In [8]:
def humanizer_agent(section):

    prompt = f"""
    Rewrite this content so it sounds
    naturally human-written.

    Requirements:
    - Vary sentence lengths
    - Mix short and long sentences
    - Avoid repetitive transitions
    - Add natural phrasing
    - Use realistic examples
    - Avoid robotic wording
    - Keep professional tone
    - Maintain same meaning
    - Keep same length
    - Do not sound overly polished
    - Make writing feel natural

    Avoid phrases like:
    "In today's world"
    "Furthermore"
    "Moreover"
    "In conclusion"

    Section:
    {section}
    """

    return safe_invoke(prompt)

In [9]:
def quality_checker_agent(article):

    prompt = f"""
    You are an expert editor.

    Analyze this article.

    Check:

    1. Content quality
    2. Repetition
    3. Logical flow
    4. Missing topics
    5. Readability
    6. Professional tone

    Give:

    - Strengths
    - Weaknesses
    - Suggestions
    - Score out of 10

    Article:
    {article}
    """

    return safe_invoke(prompt)

In [10]:
def generate_article(topic):

    print("Generating outline...")

    outline = outline_agent(topic)

    headings = [
        h.strip()
        for h in outline.split("\n")
        if h.strip()
    ]

    article = ""

    for heading in headings:

        print(f"\nResearching: {heading}")

        research = research_agent(
            topic,
            heading
        )

        print(f"Writing: {heading}")

        section = writer_agent(
            topic,
            heading,
            research,
            article
        )

        print(f"Humanizing: {heading}")

        section = humanizer_agent(
            section
        )

        article += f"\n\n{heading}\n\n"
        article += section

    return article

In [11]:
article = generate_article(
    "Blockchain in Supply Chain"
)

print(article[:3000])

Generating outline...

Researching: 1. Introduction to Blockchain in Supply Chain
Writing: 1. Introduction to Blockchain in Supply Chain
Humanizing: 1. Introduction to Blockchain in Supply Chain

Researching: 2. Understanding Supply Chain Management Challenges
Writing: 2. Understanding Supply Chain Management Challenges
Humanizing: 2. Understanding Supply Chain Management Challenges

Researching: 3. Blockchain Fundamentals and Principles
Writing: 3. Blockchain Fundamentals and Principles
Humanizing: 3. Blockchain Fundamentals and Principles

Researching: 4. Applications of Blockchain in Supply Chain Management
Writing: 4. Applications of Blockchain in Supply Chain Management
Humanizing: 4. Applications of Blockchain in Supply Chain Management

Researching: 5. Benefits of Implementing Blockchain in Supply Chain
Writing: 5. Benefits of Implementing Blockchain in Supply Chain
Humanizing: 5. Benefits of Implementing Blockchain in Supply Chain

Researching: 6. Case Studies and Real-World Ex

In [12]:
def word_count(text):
    return len(text.split())


print(
    "Total Words:",
    word_count(article)
)

Total Words: 3346


In [13]:
quality_report = quality_checker_agent(
    article
)

print(quality_report)

**Analysis of the Article**

### Content Quality
The article provides a comprehensive overview of blockchain technology and its applications in supply chain management. It covers various aspects, including the benefits, challenges, and future trends of blockchain adoption in supply chains. The content is rich in examples, case studies, and statistics, making it engaging and informative.

### Repetition
The article does contain some repetition, particularly in the introduction and conclusion sections, where similar points are made about the potential of blockchain to transform supply chain management. Additionally, some paragraphs in different sections cover similar topics, such as the benefits of blockchain in supply chain management. However, the repetition is not excessive, and the article generally provides a good balance of new information and summaries of key points.

### Logical Flow
The article is well-structured and easy to follow, with a logical flow of ideas from one section 

In [14]:
with open(
    "final_article.txt",
    "w",
    encoding="utf-8"  # used for the symbols , emojis, etc
) as file:

    file.write(article)

print("Final article saved!")

Final article saved!
